In [ ]:
import os
import re
import sys
import logging
import json
import numpy as np
import pandas as pd
import pickle
import xgboost as xgb
import plotly.graph_objects as go
from collections import defaultdict
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

from typing import Optional
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import sys
import joblib
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sqlalchemy import create_engine
sys.path.append("YOUR-PATH/actionable-hypotension/")



# ----------------------
# Configuration
# ----------------------
MODELS_DIR = "YOUR-PATH/actionable-hypotension/models_given/calibrated"
SUBANALYSIS_DIR = "YOUR-PATH/actionable-hypotension/extended_evaluation_review/results"
LOG_LEVEL = logging.INFO
# ----------------------
# Set up logging
# ----------------------
logging.basicConfig(
level=LOG_LEVEL,
format="%(asctime)s [%(levelname)s] %(message)s",
datefmt="%Y-%m-%d %H:%M:%S")
# -----
# DB connection
DATABASE_URI = "postgresql+psycopg2://USER@localhost:5434/mimic"
engine = create_engine(DATABASE_URI, future=True)

decision_thresholds = {
"mix": 0.0011452179169282317,
"inv_model": 0.0020985996816307306,
"noninv_model": 0.0008126477478072047}

UNCALIBRATED_MODELS_DIR = "YOUR-PATH/actionable-hypotension/models_given/uncalibrated"
CALIBRATED_MODELS_DIR = "YOUR-PATH/actionable-hypotension/extended_evaluation_review/models_calibrated_unbundled"

In [ ]:
def escape_for_latex(s):
    return str(s).replace('_', r'\_').replace('%', r'\%') \
                 .replace('#', r'\#') \
                 .replace('<', r'$<$').replace('>', r'$>$') \
                 .replace('[', r'[').replace(']', r']') \
                 .replace('=', r'=')  # = ist meist OK außerhalb Mathemodus

def shorten_count_str(count_str):
    try:
        pos, total = count_str.strip().split('/')
        total = int(total)

        if total < 1000:
            total_str = str(total)
        elif total < 100000:
            total_str = f"{round(total / 1000, 1)}K".replace(".0", "")
        else:
            total_str = f"{round(total / 1000)}K"

        return f"{pos}/{total_str}"
    except Exception:
        return count_str


def format_latex_row(row: str):
    """
    Input:
    'All patients,0.823 [0.814–0.831],2256/1116174,0.853,0.612'

    Output:
    'All patients & 0.823 [0.814–0.831] & 0.853 & 0.612 & 2256/1116K'
    """
    try:
        parts = [p.strip() for p in row.split(",")]
        if len(parts) < 5:
            return row  # fallback

        subgroup, auc, count_str, sens, spec = parts[:5]

        count_short = shorten_count_str(count_str)

        return f"{subgroup} & {auc} & {sens} & {spec} & {count_short}"

    except Exception:
        return row


mix_biometric_translation = {
    "age_bin": {"1": "< 53.0", "2": "[53.0 - 64.5)", "3": "[64.5 - 76.0)", "4": ">= 76.0"},
    "height_bin": {"1": "< 163.0", "2": "[163.0 - 170.5)", "3": "[170.5 - 178.0)", "4": ">= 178.0"},
    "weight_bin": {"1": "< 65.6", "2": "[65.6 - 79.6)", "3": "[79.6 - 93.6)", "4": ">= 93.6"},
    "bmi_bin": {"1": "< 23.4", "2": "[23.4 - 27.7)", "3": "[27.7 - 32.1)", "4": ">= 32.1"}
}

dem_translation = {
    "gender_bin": {"0": "Male", "1": "Female"},
    "ethnicity_bin": {"0": "White", "1": "Black", "2": "Asian", "3": "Hispanic", "4": "Other"}
}


def translate_bin_names(df):
    def translate_feature(feature):
        match = re.match(r"(\w+_bin) = (\d)", str(feature))
        if match:
            bin_name, bin_value = match.groups()
            if bin_name in mix_biometric_translation:
                label = mix_biometric_translation[bin_name].get(bin_value, bin_value)
                return f"{bin_name[:-4]} {label}"
            elif bin_name in dem_translation:
                label = dem_translation[bin_name].get(bin_value, bin_value)
                return f"{bin_name[:-4]}: {label}"
        return feature

    df['Feature'] = df['Feature'].apply(translate_feature)
    return df


def bold_best_auc_in_row(row: str) -> str:
    blocks = [b.strip() for b in row.split("&")]

    # Identify AUC columns: 1, 5, 9, ... (i.e. every 4th after feature)
    auc_indices = list(range(1, len(blocks), 4))
    aucs = []

    for idx in auc_indices:
        match = re.match(r"([0-9]+\.[0-9]+)", blocks[idx])
        if match:
            aucs.append((idx, float(match.group(1))))

    if not aucs:  # no AUCs found → return unchanged
        return row

    # Find best AUC(s) (support ties)
    best_val = max(val for _, val in aucs)
    best_indices = [idx for idx, val in aucs if val == best_val]

    # Bold all best AUCs
    for idx in best_indices:
        auc_str = re.match(r"([0-9]+\.[0-9]+)", blocks[idx]).group(1)
        blocks[idx] = blocks[idx].replace(
            auc_str, f"\\textbf{{{auc_str}}}", 1
        )

    return " & ".join(blocks)


In [ ]:
import pandas as pd

df = pd.read_csv(
    "YOUR-PATH/actionable-hypotension/extended_evaluation_review/results/extended_subgroup_analysis_v4.csv"
)

# shorten counts
df["pos_total_short"] = df["pos/total"].apply(shorten_count_str)

# clean column names (important — avoids hidden leading spaces)
df.columns = df.columns.str.strip()

# ensure numeric + round
df["Sensitivity"] = df["Sensitivity"].apply(lambda x: f"{x:.3f}")
df["Specificity"] = df["Specificity"].apply(lambda x: f"{x:.3f}")

df_latex = pd.DataFrame({
    "Subgroup": df["Subgroup"],
    "AUROC [CI]": df["AUROC [CI]"],
    "pos/total": df["pos_total_short"],
    "Sensitivity": df["Sensitivity"],
    "Specificity": df["Specificity"]
})

latex_code = df_latex.to_latex(index=False, escape=False)

final_latex = "\\begin{scriptsize}\n" + latex_code + "\n\\end{scriptsize}"

print(final_latex)